# 🎓 AI University Knowledge & Study Assistant

This notebook combines the supplied code into one project. It keeps the original Mistral generation function, FAISS + HuggingFace embeddings + PyPDFLoader RAG pipeline, the custom LangChain LLM/chaining pattern, and the StructuredOutputParser pattern.

The notebook creates `app.py` and then launches it with Streamlit.


## 1. Install dependencies

Run this cell once. The packages are based on the libraries already used in the supplied notebooks, with Streamlit added for the application.

In [1]:
!pip install -U streamlit faiss-cpu langchain langchain-community langchain-core langchain-huggingface langchain-classic langchain-text-splitters pypdf sentence-transformers transformers==4.52.4 accelerate torch -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 98.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 69.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 72.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.8/572.8 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 3.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 10.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 8.6 MB

## 2. Core model code

This preserves the `generate_text()` approach from the supplied Mistral notebook.

In [2]:
!pip install --force-reinstall --no-deps torch==2.11.0 torchvision==0.26.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 70.2 MB/s eta 0:00:00:00:0100:01


In [3]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = os.getenv('MODEL_NAME', 'mistralai/Mistral-7B-Instruct-v0.2')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map='auto'
).eval()

def generate_text(prompt, max_length=1000, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_return_sequences=num_return_sequences,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
        )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs][0]


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

## 3. LangChain custom LLM + chaining

This adapts the `CustomHFLLM` and `LLMChain` pattern from `Update_Chaining.ipynb`.

In [4]:
from typing import Any
from langchain_core.language_models.llms import LLM
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain

class CustomHFLLM(LLM):
    def _call(self, prompt: str, stop: Any = None) -> str:
        return generate_text(prompt, max_length=1000)

    @property
    def _llm_type(self) -> str:
        return 'custom_huggingface'

llm = CustomHFLLM()

qa_prompt = PromptTemplate(
    input_variables=['context', 'question'],
    template='''You are an AI University Knowledge & Study Assistant.

Answer using ONLY the provided university/course context.
If the answer is not contained in the context, say that you could not find it in the uploaded documents.

Context:
{context}

Question:
{question}

Answer:'''
)

qa_chain = LLMChain(llm=llm, prompt=qa_prompt)


/tmp/ipykernel_49/1818432872.py:32: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  qa_chain = LLMChain(llm=llm, prompt=qa_prompt)


## 4. RAG document processing

This preserves your `PyPDFLoader`, `CharacterTextSplitter`, `HuggingFaceEmbeddings`, and `FAISS.from_documents()` approach.

In [5]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter

embedding_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
embedding = HuggingFaceEmbeddings(model_name=embedding_model_name)

def build_vector_database(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    chunks = text_splitter.split_documents(documents)
    vectordb = FAISS.from_documents(chunks, embedding)
    return vectordb, documents, chunks


/tmp/ipykernel_49/2536412065.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 5. RAG question answering

This keeps the original similarity-search logic (`k=3`) but sends the retrieved context through the LangChain chain.

In [6]:
def ask_question(vectordb, query):
    docs = vectordb.similarity_search(query, k=3)
    context = '\n\n'.join([doc.page_content for doc in docs])
    result = qa_chain.run({'context': context, 'question': query})
    return result.strip(), docs


## 6. Structured output parser for MCQs

This uses the same `ResponseSchema`, `StructuredOutputParser`, format instructions, JSON extraction, and parsing pattern from `Update_OutputParser.ipynb`, adapted to university MCQs.

In [7]:
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
import re

mcq_schema = ResponseSchema(
    name='questions',
    description='A list of MCQs. Each item contains question, options A-D, correct_answer, and explanation.'
)
mcq_parser = StructuredOutputParser.from_response_schemas([mcq_schema])
format_instructions = mcq_parser.get_format_instructions()

def extract_json_block(text):
    pattern = r'```json\\s*(.*?)\\s*```'
    matches = re.findall(pattern, text, re.DOTALL)
    if matches:
        return f'```json\\n{matches[-1]}\\n```'
    return text.strip()


## 7. Create the Streamlit application

The following cell writes the complete `app.py` file. The application contains PDF upload, FAISS retrieval, grounded Q&A, summaries, and structured MCQ generation.

In [8]:
APP_CODE = 'import os\nimport re\nimport tempfile\nfrom typing import Any, List\n\nimport streamlit as st\nimport torch\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\nfrom langchain_community.vectorstores import FAISS\nfrom langchain_huggingface import HuggingFaceEmbeddings\nfrom langchain_community.document_loaders import PyPDFLoader\nfrom langchain_text_splitters import CharacterTextSplitter\nfrom langchain_core.language_models.llms import LLM\nfrom langchain_core.prompts import PromptTemplate\nfrom langchain_classic.chains import LLMChain\nfrom langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema\n\n\n# ============================================================\n# 1. MODEL — based on your original Mistral generate_text code\n# ============================================================\n\nMODEL_NAME = os.getenv("MODEL_NAME", "mistralai/Mistral-7B-Instruct-v0.2")\nEMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"\n\n\n@st.cache_resource(show_spinner="Loading Mistral model...")\ndef load_model():\n    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)\n\n    # Your original code uses float16 + device_map="auto".\n    # CPU-only machines cannot normally run this 7B model efficiently.\n    dtype = torch.float16 if torch.cuda.is_available() else torch.float32\n\n    model = AutoModelForCausalLM.from_pretrained(\n        MODEL_NAME,\n        torch_dtype=dtype,\n        device_map="auto",\n    )\n    model.eval()\n    return tokenizer, model\n\n\ndef generate_text(prompt, max_length=1000, num_return_sequences=1):\n    tokenizer, model = load_model()\n\n    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)\n\n    with torch.no_grad():\n        outputs = model.generate(\n            **inputs,\n            max_length=max_length,\n            num_return_sequences=num_return_sequences,\n            do_sample=True,\n            top_k=50,\n            top_p=0.95,\n            temperature=0.7,\n            pad_token_id=tokenizer.eos_token_id,\n        )\n\n    return [\n        tokenizer.decode(output, skip_special_tokens=True)\n        for output in outputs\n    ][0]\n\n\n# ============================================================\n# 2. LANGCHAIN CUSTOM LLM — based on your Chaining notebook\n# ============================================================\n\nclass CustomHFLLM(LLM):\n    def _call(self, prompt: str, stop: Any = None) -> str:\n        return generate_text(prompt, max_length=1000)\n\n    @property\n    def _llm_type(self) -> str:\n        return "custom_huggingface"\n\n\n@st.cache_resource\ndef get_llm():\n    return CustomHFLLM()\n\n\n# ============================================================\n# 3. RAG DOCUMENT PROCESSING — based on your original code\n# ============================================================\n\n@st.cache_resource\ndef get_embeddings():\n    return HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)\n\n\ndef build_vector_database(uploaded_files):\n    all_documents = []\n\n    for uploaded_file in uploaded_files:\n        # PyPDFLoader works with a file path, so save the Streamlit\n        # upload temporarily.\n        with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:\n            tmp.write(uploaded_file.getvalue())\n            temp_path = tmp.name\n\n        try:\n            loader = PyPDFLoader(temp_path)\n            documents = loader.load()\n\n            # Keep the filename in metadata for source display.\n            for doc in documents:\n                doc.metadata["source_file"] = uploaded_file.name\n\n            all_documents.extend(documents)\n        finally:\n            try:\n                os.remove(temp_path)\n            except OSError:\n                pass\n\n    # Same chunking approach from your notebook.\n    text_splitter = CharacterTextSplitter(\n        chunk_size=1000,\n        chunk_overlap=100\n    )\n    chunks = text_splitter.split_documents(all_documents)\n\n    embedding = get_embeddings()\n\n    # Same FAISS.from_documents approach from your notebook.\n    vectordb = FAISS.from_documents(chunks, embedding)\n    return vectordb, len(documents), len(chunks)\n\n\n# ============================================================\n# 4. LANGCHAIN RAG CHAIN\n# ============================================================\n\nRAG_PROMPT = PromptTemplate(\n    input_variables=["context", "question"],\n    template="""You are an AI University Knowledge & Study Assistant.\n\nAnswer the user\'s question using ONLY the provided university/course context.\n\nRules:\n- Do not use outside knowledge.\n- If the answer is not contained in the context, say:\n  "I could not find this information in the uploaded documents."\n- Explain the answer clearly and at a university-student level.\n- Do not invent facts, definitions, examples, page numbers, or references.\n\nContext:\n{context}\n\nQuestion:\n{question}\n\nAnswer:"""\n)\n\n\ndef create_rag_chain():\n    llm = get_llm()\n    return LLMChain(llm=llm, prompt=RAG_PROMPT)\n\n\ndef ask_question(vectordb, query, k=3):\n    docs = vectordb.similarity_search(query, k=k)\n\n    context = "\\n\\n".join(\n        doc.page_content for doc in docs\n    )\n\n    chain = create_rag_chain()\n    result = chain.run({\n        "context": context,\n        "question": query\n    })\n\n    return result.strip(), docs\n\n\n# ============================================================\n# 5. STRUCTURED OUTPUT PARSER — based on your OutputParser notebook\n# ============================================================\n\nmcq_schemas = [\n    ResponseSchema(\n        name="questions",\n        description=(\n            "A list of multiple choice questions. Each item must contain "\n            "question, options (A-D), correct_answer, and explanation."\n        ),\n    )\n]\n\nmcq_parser = StructuredOutputParser.from_response_schemas(mcq_schemas)\nmcq_format_instructions = mcq_parser.get_format_instructions()\n\n\ndef extract_json_block(text):\n    pattern = r"```json\\s*(.*?)\\s*```"\n    matches = re.findall(pattern, text, re.DOTALL)\n\n    if matches:\n        return f"```json\\n{matches[-1]}\\n```"\n\n    # Some models return JSON without markdown fences.\n    return text.strip()\n\n\ndef generate_mcqs(vectordb, topic, number):\n    docs = vectordb.similarity_search(topic, k=5)\n    context = "\\n\\n".join(doc.page_content for doc in docs)\n\n    prompt = PromptTemplate(\n        template="""You are a university study assistant.\n\nCreate {number} difficult multiple-choice questions about:\n{topic}\n\nUse ONLY the provided course context.\n\nEach question must have:\n- question\n- four options labeled A, B, C, D\n- correct_answer\n- explanation\n\nDo not create information that is not supported by the context.\n\nReturn ONLY JSON according to these instructions:\n{format_instructions}\n\nCourse context:\n{context}\n""",\n        input_variables=[\n            "number",\n            "topic",\n            "format_instructions",\n            "context",\n        ],\n    ).format(\n        number=number,\n        topic=topic,\n        format_instructions=mcq_format_instructions,\n        context=context,\n    )\n\n    raw = generate_text(prompt, max_length=1800)\n    json_text = extract_json_block(raw)\n\n    try:\n        parsed = mcq_parser.parse(json_text)\n        return parsed\n    except Exception:\n        # Return raw output instead of hiding the model/parser error.\n        return {"raw_output": raw}\n\n\n# ============================================================\n# 6. SUMMARY\n# ============================================================\n\ndef generate_summary(vectordb, topic):\n    docs = vectordb.similarity_search(topic, k=5)\n    context = "\\n\\n".join(doc.page_content for doc in docs)\n\n    prompt = f"""You are a university study assistant.\n\nSummarize the important concepts about "{topic}" using ONLY the\nuploaded course material below.\n\nOrganize the summary as:\n1. Main concepts\n2. Important definitions\n3. Key points\n4. Important relationships or steps\n5. Short revision checklist\n\nDo not add information that is not in the context.\n\nCourse context:\n{context}\n\nSummary:\n"""\n    return generate_text(prompt, max_length=1600).strip()\n\n\n# ============================================================\n# 7. STREAMLIT UI\n# ============================================================\n\nst.set_page_config(\n    page_title="AI University Study Assistant",\n    page_icon="🎓",\n    layout="wide",\n)\n\nst.title("🎓 AI University Knowledge & Study Assistant")\nst.write(\n    "Upload your university PDFs, then ask questions, generate summaries, "\n    "or create difficult MCQs grounded in the uploaded material."\n)\n\nwith st.sidebar:\n    st.header("📚 Course Documents")\n\n    uploaded_files = st.file_uploader(\n        "Upload PDF lecture/course files",\n        type=["pdf"],\n        accept_multiple_files=True,\n    )\n\n    if uploaded_files:\n        st.write(f"**{len(uploaded_files)} file(s) selected**")\n        for file in uploaded_files:\n            st.caption(f"• {file.name}")\n\n        if st.button("🔄 Build Knowledge Base", use_container_width=True):\n            with st.spinner("Loading, splitting and embedding documents..."):\n                try:\n                    vectordb, pages, chunks = build_vector_database(\n                        uploaded_files\n                    )\n                    st.session_state.vectordb = vectordb\n                    st.session_state.files = [\n                        f.name for f in uploaded_files\n                    ]\n                    st.success(\n                        f"Knowledge base ready: {pages} pages, "\n                        f"{chunks} chunks."\n                    )\n                except Exception as e:\n                    st.error(f"Could not process PDFs: {e}")\n\n    st.divider()\n    st.caption("Powered by LangChain + FAISS + Hugging Face + Mistral")\n\n\nif "vectordb" not in st.session_state:\n    st.info(\n        "👈 Upload one or more PDF course documents and click "\n        "\'Build Knowledge Base\' to start."\n    )\nelse:\n    tab1, tab2, tab3 = st.tabs(\n        ["💬 Ask Questions", "📝 Summarize", "🧠 Generate MCQs"]\n    )\n\n    # --------------------------------------------------------\n    # Q&A\n    # --------------------------------------------------------\n    with tab1:\n        st.subheader("Ask about your course")\n\n        question = st.text_area(\n            "Question",\n            placeholder=(\n                "Example: Explain normalization based only on "\n                "the uploaded lectures."\n            ),\n        )\n\n        if st.button("Ask", type="primary"):\n            if not question.strip():\n                st.warning("Please enter a question.")\n            else:\n                with st.spinner("Searching the course material..."):\n                    try:\n                        answer, source_docs = ask_question(\n                            st.session_state.vectordb,\n                            question,\n                        )\n\n                        st.markdown("### Answer")\n                        st.write(answer)\n\n                        st.markdown("### Retrieved Sources")\n                        shown = set()\n\n                        for doc in source_docs:\n                            filename = doc.metadata.get(\n                                "source_file", "Unknown file"\n                            )\n                            page = doc.metadata.get(\n                                "page", "Unknown"\n                            )\n\n                            key = (filename, page)\n                            if key not in shown:\n                                st.caption(\n                                    f"📄 {filename} — page {page + 1 if isinstance(page, int) else page}"\n                                )\n                                shown.add(key)\n\n                    except Exception as e:\n                        st.error(f"Error: {e}")\n\n    # --------------------------------------------------------\n    # SUMMARY\n    # --------------------------------------------------------\n    with tab2:\n        st.subheader("Generate a study summary")\n\n        summary_topic = st.text_input(\n            "Topic or chapter",\n            placeholder="Example: Database normalization",\n        )\n\n        if st.button("Generate Summary"):\n            if not summary_topic.strip():\n                st.warning("Please enter a topic.")\n            else:\n                with st.spinner("Generating summary from the course material..."):\n                    try:\n                        summary = generate_summary(\n                            st.session_state.vectordb,\n                            summary_topic,\n                        )\n                        st.markdown(summary)\n                    except Exception as e:\n                        st.error(f"Error: {e}")\n\n    # --------------------------------------------------------\n    # MCQS\n    # --------------------------------------------------------\n    with tab3:\n        st.subheader("Generate difficult MCQs")\n\n        mcq_topic = st.text_input(\n            "Topic",\n            placeholder="Example: SQL normalization",\n        )\n\n        number = st.slider(\n            "Number of questions",\n            min_value=1,\n            max_value=20,\n            value=10,\n        )\n\n        if st.button("Generate MCQs"):\n            if not mcq_topic.strip():\n                st.warning("Please enter a topic.")\n            else:\n                with st.spinner("Generating questions..."):\n                    try:\n                        result = generate_mcqs(\n                            st.session_state.vectordb,\n                            mcq_topic,\n                            number,\n                        )\n\n                        if "questions" in result:\n                            for i, item in enumerate(\n                                result["questions"], 1\n                            ):\n                                st.markdown(\n                                    f"### Question {i}"\n                                )\n                                st.write(\n                                    item.get(\n                                        "question",\n                                        ""\n                                    )\n                                )\n\n                                options = item.get(\n                                    "options", {}\n                                )\n\n                                if isinstance(options, dict):\n                                    for key, value in options.items():\n                                        st.write(\n                                            f"**{key}.** {value}"\n                                        )\n                                elif isinstance(options, list):\n                                    for option in options:\n                                        st.write(f"• {option}")\n\n                                with st.expander(\n                                    "Show answer and explanation"\n                                ):\n                                    st.write(\n                                        "**Correct answer:** "\n                                        + str(\n                                            item.get(\n                                                "correct_answer",\n                                                ""\n                                            )\n                                        )\n                                    )\n                                    st.write(\n                                        "**Explanation:** "\n                                        + str(\n                                            item.get(\n                                                "explanation",\n                                                ""\n                                            )\n                                        )\n                                    )\n                        else:\n                            st.warning(\n                                "The structured output parser could not "\n                                "parse the model response."\n                            )\n                            st.code(\n                                result.get("raw_output", ""),\n                                language="text",\n                            )\n\n                    except Exception as e:\n                        st.error(f"Error: {e}")\n'
with open('app.py', 'w', encoding='utf-8') as f:
    f.write(APP_CODE)

print('Created app.py successfully.')


Created app.py successfully.


## 8. Run the application with Streamlit

Run this cell after `app.py` has been created. Streamlit starts the web application on port 8501.

In [9]:
""""import subprocess
import sys

streamlit_process = subprocess.Popen([
    sys.executable, '-m', 'streamlit', 'run', 'app.py',
    '--server.port', '8501',
    '--server.address', '0.0.0.0',
    '--server.headless', 'true'
])

print('Streamlit is running on port 8501.')
print('If your notebook environment provides a proxy/preview, open the Streamlit port there.')"""


'"import subprocess\nimport sys\n\nstreamlit_process = subprocess.Popen([\n    sys.executable, \'-m\', \'streamlit\', \'run\', \'app.py\',\n    \'--server.port\', \'8501\',\n    \'--server.address\', \'0.0.0.0\',\n    \'--server.headless\', \'true\'\n])\n\nprint(\'Streamlit is running on port 8501.\')\nprint(\'If your notebook environment provides a proxy/preview, open the Streamlit port there.\')'

In [10]:
!pip install pyngrok -q
from pyngrok import ngrok
import os
import subprocess
import sys

# Set your Ngrok Token here
NGROK_TOKEN = "3KJq6MYXMGPzY5TBGx9JPcR5F6W_5vvetK1voXpU9eP7xTbg3"
ngrok.set_auth_token(NGROK_TOKEN)

# Start Streamlit in the background
streamlit_process = subprocess.Popen([
    sys.executable, '-m', 'streamlit', 'run', 'app.py',
    '--server.port', '8501',
    '--server.address', '0.0.0.0',
    '--server.headless', 'true'
])

# Generate public URL
public_url = ngrok.connect(8501).public_url
print('👉 Open your Streamlit App here:', public_url)

👉 Open your Streamlit App here: https://moody-citrus-pretext.ngrok-free.dev                        


## Optional: expose the Streamlit app through ngrok

This is optional. It follows the ngrok approach from the supplied FastAPI notebook. Set `NGROK_TOKEN` before running it.

In [11]:
# Optional
#!pip install pyngrok -q
# from pyngrok import ngrok
# import os
# ngrok.set_auth_token(os.environ['NGROK_TOKEN'])
# public_url = ngrok.connect(8501).public_url
# print('Public Streamlit URL:', public_url)
